In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
books = pd.read_csv('../data/processed/books_clean.csv')
ratings = pd.read_csv('../data/processed/ratings_clean.csv')
explicit_ratings = pd.read_csv(
    '../data/processed/explicit_ratings.csv'
)
ratings_with_books = pd.read_csv(
    '../data/processed/ratings_with_books.csv'
)

In [ ]:
print("Books:", books.shape)
print("Ratings:", ratings.shape)
print("Explicit ratings:", explicit_ratings.shape)


In [ ]:

books['isbn'] = books['isbn'].astype(str)
ratings['isbn'] = ratings['isbn'].astype(str)
explicit_ratings['isbn'] = explicit_ratings['isbn'].astype(str)


In [ ]:

plt.figure(figsize=(10, 5))

sns.countplot(
    data=explicit_ratings,
    x='rating',
    color='steelblue'
)

plt.title('Distribution of Explicit Book Ratings')
plt.xlabel('Rating')
plt.ylabel('Number of Ratings')
plt.show()


In [ ]:

book_rating_counts = (
    ratings_with_books
    .groupby('title')['rating']
    .count()
    .sort_values(ascending=False)
)

book_rating_counts.head(10)


In [ ]:

plt.figure(figsize=(11, 6))

book_rating_counts.head(10).sort_values().plot(
    kind='barh'
)

plt.title('Top 10 Most Rated Books')
plt.xlabel('Number of Explicit Ratings')
plt.ylabel('Book Title')
plt.tight_layout()
plt.show()


In [ ]:

book_stats = (
    ratings_with_books
    .groupby('title')
    .agg(
        num_ratings=('rating', 'count'),
        avg_rating=('rating', 'mean')
    )
    .reset_index()
)

book_stats.head()


In [ ]:

popular_books = book_stats[
    book_stats['num_ratings'] >= 50
].sort_values(
    'avg_rating',
    ascending=False
)

popular_books.head(10)


In [ ]:

user_activity = (
    explicit_ratings
    .groupby('user_id')['isbn']
    .nunique()
)

user_activity.describe()


In [ ]:

plt.figure(figsize=(10, 5))

sns.histplot(
    user_activity,
    bins=50
)

plt.title('Distribution of User Rating Activity')
plt.xlabel('Number of Books Rated')
plt.ylabel('Number of Users')
plt.xlim(0, 100)
plt.show()


In [ ]:

n_users = explicit_ratings['user_id'].nunique()
n_books = explicit_ratings['isbn'].nunique()
n_ratings = len(explicit_ratings)

sparsity = 1 - (
    n_ratings / (n_users * n_books)
)

print("Unique users:", n_users)
print("Unique books:", n_books)
print(f"Sparsity: {sparsity:.2%}")


In [ ]:

from pathlib import Path

output_dir = Path('../data/processed')
output_dir.mkdir(parents=True, exist_ok=True)

book_stats.to_csv(
    output_dir / 'book_stats.csv',
    index=False
)

print("EDA statistics saved!")
